In [ ]:
import pandas as pd
import numpy as np
import random
import string
import hashlib
from datetime import datetime, timedelta


SEED = 99
random.seed(SEED)
np.random.seed(SEED)

INPUT_CSV = "transactions_final.csv"
OUTPUT_CSV = "transactions_final_with_typologies.csv"

# Read everything as text so we preserve account numbers / IDs exactly.
df = pd.read_csv("./aml_synthetic_data/transactions_final.csv", dtype=str, keep_default_na=False)

if "transaction_id" not in df.columns:
    raise ValueError("transaction_id column not found in input file.")

# Ensure the target label columns exist
for col in ["is_aml", "aml_typology", "typology_group_id"]:
    if col not in df.columns:
        df[col] = ""

# Base rows and account profiles
base_rows = df.to_dict("records")
existing_ids = set(df["transaction_id"].astype(str).tolist())

def _norm(v, default=""):
    if v is None:
        return default
    v = str(v)
    return default if v.lower() == "nan" else v

def _parse_dt(date_str):
    if not date_str:
        return None
    for fmt in ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%Y/%m/%d"):
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            pass
    return None

dates = pd.to_datetime(df["datestamp"], dayfirst=True, errors="coerce")
MIN_DATE = dates.min()
MAX_DATE = dates.max()
if pd.isna(MIN_DATE) or pd.isna(MAX_DATE):
    MIN_DATE = pd.Timestamp("2024-01-01")
    MAX_DATE = pd.Timestamp("2025-03-31")
else:
    MIN_DATE = MIN_DATE.normalize()
    MAX_DATE = MAX_DATE.normalize()

def random_date_between(start_ts=MIN_DATE, end_ts=MAX_DATE):
    span = max(int((end_ts - start_ts).days), 1)
    dt = start_ts + pd.Timedelta(days=random.randint(0, span))
    return dt.to_pydatetime()

def random_time_on_date(date_obj, preferred_hour=None, minute=None, second=None):
    hour = int(preferred_hour) if preferred_hour is not None else random.randint(0, 23)
    minute = random.randint(0, 59) if minute is None else minute
    second = random.randint(0, 59) if second is None else second
    return datetime(date_obj.year, date_obj.month, date_obj.day, hour, minute, second)

def fmt_date(dt):
    return dt.strftime("%d/%m/%Y")

def fmt_time(dt):
    return dt.strftime("%H:%M:%S")

def new_txn_id():
    while True:
        txid = "TXN" + "".join(random.choices(string.ascii_uppercase + string.digits, k=16))
        if txid not in existing_ids:
            existing_ids.add(txid)
            return txid

def is_business_profile(row):
    ct = _norm(row.get("customer_type", "")).lower()
    cat = _norm(row.get("account_category", "")).lower()
    ent = _norm(row.get("customer_entity_type", "")).lower()
    return (
        ct in {"non-individual", "business", "entity", "corporate"} or
        cat in {"current", "business", "corporate"} or
        ent in {"llp", "partnership", "private limited", "public limited", "trust", "society", "huf", "government body"}
    )

def acct_key(value):
    return _norm(value).strip()

acct_profiles = {}
for r in base_rows:
    acct = acct_key(r.get("customer_account_number", ""))
    if acct and acct not in acct_profiles:
        acct_profiles[acct] = r

all_accounts = list(acct_profiles.keys())

def choose_accounts(n, exclude=None, predicate=None):
    exclude = set(acct_key(x) for x in (exclude or []))
    pool = []
    for acct in all_accounts:
        if acct in exclude:
            continue
        prof = acct_profiles.get(acct, {})
        if predicate and not predicate(prof):
            continue
        pool.append(acct)
    if not pool:
        return []
    if n >= len(pool):
        return random.sample(pool, len(pool))
    return random.sample(pool, n)

def build_row(
    sender_acct,
    receiver_acct,
    amount,
    dt,
    typology,
    group_id,
    channel="NEFT",
    dr_cr="Dr",
    sender_country="IN",
    receiver_country="IN",
    is_aml=1,
    balance_before=None,
    balance_after=None,
    device_mode="sender",
    shared_device_id=None,
    ip_mode="sender",
    shared_ip=None,
    extra_updates=None,
):
    sender_acct = acct_key(sender_acct)
    receiver_acct = acct_key(receiver_acct)
    sender_profile = acct_profiles.get(sender_acct, random.choice(base_rows))
    receiver_profile = acct_profiles.get(receiver_acct, sender_profile)

    row = dict(sender_profile)

    # Core transaction values
    row["transaction_id"] = new_txn_id()
    row["timestamp"] = fmt_time(dt)
    row["datestamp"] = fmt_date(dt)
    row["transaction_amount"] = f"{float(amount):.2f}"
    row["transaction_mode_channel_bank"] = channel
    row["transaction_status"] = "Success"
    row["refund_chargeback_flag"] = "N"
    row["transaction_type_dr_cr"] = dr_cr
    row["cash_flag"] = "Y" if channel in {"Branch Cash", "ATM"} else "N"

    # Party fields
    row["customer_account_number"] = sender_acct
    row["counterparty_account_number"] = receiver_acct
    row["customer_name"] = _norm(sender_profile.get("customer_name", row.get("customer_name", "")))
    row["counterparty_name"] = _norm(receiver_profile.get("customer_name", f"External {receiver_acct}"))
    row["customer_cif_id"] = _norm(sender_profile.get("customer_cif_id", row.get("customer_cif_id", "")))
    row["counterparty_branch_ifsc_swift"] = _norm(receiver_profile.get("customer_branch_ifsc_code", row.get("counterparty_branch_ifsc_swift", "")))

    # Country / geo
    row["sender_country_code"] = sender_country
    row["receiver_country_code"] = receiver_country
    if sender_profile.get("geo_location_city_country"):
        row["geo_location_city_country"] = sender_profile.get("geo_location_city_country", row.get("geo_location_city_country", ""))
    if sender_profile.get("gps_coordinates_lat"):
        row["gps_coordinates_lat"] = sender_profile.get("gps_coordinates_lat", row.get("gps_coordinates_lat", ""))
    if sender_profile.get("gps_coordinates_lon"):
        row["gps_coordinates_lon"] = sender_profile.get("gps_coordinates_lon", row.get("gps_coordinates_lon", ""))

    # Device / session / network
    if shared_device_id is not None:
        row["device_id_fingerprint"] = shared_device_id
    else:
        if device_mode == "sender":
            row["device_id_fingerprint"] = _norm(sender_profile.get("device_id_fingerprint", row.get("device_id_fingerprint", "")))
        elif device_mode == "shared":
            row["device_id_fingerprint"] = hashlib.md5(f"{sender_acct}|{group_id}".encode()).hexdigest()[:32]
        else:
            row["device_id_fingerprint"] = hashlib.md5(f"{sender_acct}|{group_id}|{random.random()}".encode()).hexdigest()[:32]

    if shared_ip is not None:
        row["ip_address"] = shared_ip
    else:
        if ip_mode == "sender":
            row["ip_address"] = _norm(sender_profile.get("ip_address", row.get("ip_address", "")))
        elif ip_mode == "shared":
            row["ip_address"] = f"103.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"
        else:
            row["ip_address"] = f"{random.randint(11,223)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

    row["session_id"] = hashlib.md5(f"{sender_acct}|{receiver_acct}|{dt.isoformat()}|{random.random()}".encode()).hexdigest()[:24]
    row["authentication_method"] = _norm(sender_profile.get("authentication_method", row.get("authentication_method", random.choice(["OTP", "MPIN", "Password"]))))

    # AML labels
    row["is_aml"] = str(int(is_aml))
    row["aml_typology"] = typology if int(is_aml) == 1 else ""
    row["typology_group_id"] = group_id if int(is_aml) == 1 else ""

    # Optional balance fields
    if balance_before is not None:
        row["wallet_balance_before"] = f"{float(balance_before):.2f}"
    if balance_after is not None:
        row["wallet_balance_after"] = f"{float(balance_after):.2f}"

    if extra_updates:
        for k, v in extra_updates.items():
            if k in row:
                row[k] = v

    return row



In [ ]:
# ------------------------------------------------------------
# Typology 6: Burst Activity After Dormancy in a Cluster
# ------------------------------------------------------------#
def inject_typology_6():
    out = []
    severities = [("base", 6), ("obfuscated", 4), ("worst", 3)]
    for sev, n_scenarios in severities:
        for i in range(n_scenarios):
            gid = f"T06_{sev[:1].upper()}{i+1:04d}"
            cluster_size = {"base": 4, "obfuscated": 6, "worst": 8}[sev]
            cluster = choose_accounts(cluster_size, predicate=None)
            if len(cluster) < 3:
                continue

            exit_accounts = choose_accounts(2 if sev != "worst" else 3, exclude=cluster)
            if not exit_accounts:
                exit_accounts = cluster[:1]

            burst_date = random_date_between()
            quiet_days = {"base": 20, "obfuscated": 30, "worst": 45}[sev]
            burst_days = {"base": 1, "obfuscated": 2, "worst": 3}[sev]

            # Supporting low-activity rows before the burst (non-AML)
            for j, acct in enumerate(cluster[:max(2, cluster_size // 2)]):
                quiet_dt = burst_date - timedelta(days=random.randint(quiet_days // 2, quiet_days))
                quiet_row = build_row(
                    sender_acct=acct,
                    receiver_acct=acct,
                    amount=random.uniform(250, 2500),
                    dt=random_time_on_date(quiet_dt, preferred_hour=random.choice([10, 11, 14, 16])),
                    typology="",
                    group_id="",
                    channel=random.choice(["UPI", "IMPS", "Mobile Banking"]),
                    dr_cr="Dr",
                    is_aml=0,
                    device_mode="sender",
                    ip_mode="sender",
                )
                out.append(quiet_row)

            # Burst window: dense transaction burst
            shared_device = hashlib.md5(f"BURST|{gid}".encode()).hexdigest()[:32] if sev != "base" else None
            shared_ip = f"103.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}" if sev == "worst" else None

            burst_txn_count = {"base": 8, "obfuscated": 14, "worst": 24}[sev]
            for k in range(burst_txn_count):
                sender = random.choice(cluster)
                if sev == "worst" and k % 3 == 0:
                    receiver = random.choice(cluster)  # internal churn
                else:
                    receiver = random.choice(exit_accounts + cluster)

                amt_low, amt_high = {
                    "base": (15000, 65000),
                    "obfuscated": (10000, 120000),
                    "worst": (5000, 250000),
                }[sev]
                amount = random.uniform(amt_low, amt_high)

                dt = burst_date + timedelta(days=random.randint(0, burst_days))
                dt = random_time_on_date(dt, preferred_hour=random.choice([8, 9, 10, 11, 14, 18, 19, 21]))

                out.append(build_row(
                    sender_acct=sender,
                    receiver_acct=receiver,
                    amount=amount,
                    dt=dt,
                    typology="Burst Activity After Dormancy in a Cluster",
                    group_id=gid,
                    channel=random.choice(["UPI", "IMPS", "NEFT", "RTGS"]),
                    dr_cr="Dr",
                    sender_country="IN",
                    receiver_country="IN",
                    is_aml=1,
                    device_mode="shared" if shared_device else "sender",
                    shared_device_id=shared_device,
                    ip_mode="shared" if shared_ip else "sender",
                    shared_ip=shared_ip,
                    extra_updates={
                        "non_face_to_face_flag": "Y" if sev == "worst" else "N",
                        "vpn_flag": "Y" if sev == "worst" else "N",
                        "emulator_flag": "Y" if sev == "worst" else "N",
                    }
                ))
    return out



In [7]:
# ------------------------------------------------------------
# Typology 7: High-Velocity In/Out with Low Closing Balance
# ------------------------------------------------------------
def inject_typology_7():
    out = []
    severities = [("base", 6), ("obfuscated", 4), ("worst", 3)]
    for sev, n_scenarios in severities:
        for i in range(n_scenarios):
            gid = f"T07_{sev[:1].upper()}{i+1:04d}"
            bridge_candidates = choose_accounts(5 if sev != "worst" else 8, predicate=None)
            if len(bridge_candidates) < 3:
                continue

            bridge = bridge_candidates[0]
            feeders = bridge_candidates[1:3] if sev != "worst" else bridge_candidates[1:4]
            exits = choose_accounts(2 if sev != "worst" else 4, exclude=bridge_candidates)
            if not exits:
                exits = [bridge_candidates[-1]]

            dt0 = random_date_between()
            cycles = {"base": 3, "obfuscated": 5, "worst": 8}[sev]
            balance = random.uniform(200.0, 1500.0)  # low closing balance seed

            for c in range(cycles):
                feeder = random.choice(feeders)
                exit_acct = random.choice(exits)

                incoming = random.uniform(50000, 250000 if sev != "worst" else 500000)
                outgoing = incoming * random.uniform(0.94, 0.995)

                # Inflow
                dt_in = dt0 + timedelta(hours=c * 2)
                balance_before = balance
                balance_after = balance_before + incoming
                out.append(build_row(
                    sender_acct=feeder,
                    receiver_acct=bridge,
                    amount=incoming,
                    dt=random_time_on_date(dt_in, preferred_hour=random.choice([9, 10, 11, 13, 15, 18])),
                    typology="High-Velocity In/Out with Low Closing Balance",
                    group_id=gid,
                    channel=random.choice(["NEFT", "RTGS", "IMPS", "Internet Banking"]),
                    dr_cr="Cr",
                    is_aml=1,
                    balance_before=balance_before,
                    balance_after=balance_after,
                    device_mode="sender",
                    ip_mode="sender",
                    extra_updates={
                        "wallet_account_id": "",
                    }
                ))

                # Outflow within minutes / same day
                dt_out = dt_in + timedelta(minutes=random.randint(5, 90))
                balance_before = balance_after
                balance_after = max(balance_before - outgoing, random.uniform(50, 900))
                balance = balance_after  # keep low

                out.append(build_row(
                    sender_acct=bridge,
                    receiver_acct=exit_acct,
                    amount=outgoing,
                    dt=random_time_on_date(dt_out, preferred_hour=min(dt_out.hour, 23)),
                    typology="High-Velocity In/Out with Low Closing Balance",
                    group_id=gid,
                    channel=random.choice(["NEFT", "RTGS", "IMPS", "UPI"]),
                    dr_cr="Dr",
                    is_aml=1,
                    balance_before=balance_before,
                    balance_after=balance_after,
                    device_mode="shared" if sev == "worst" else "sender",
                    ip_mode="shared" if sev == "worst" else "sender",
                    extra_updates={
                        "non_face_to_face_flag": "Y" if sev == "worst" else "N",
                        "vpn_flag": "Y" if sev == "worst" else "N",
                    }
                ))
    return out



In [8]:
# ------------------------------------------------------------
# Typology 8: Multi-Bank, Multi-Jurisdiction Chains
# ------------------------------------------------------------
def inject_typology_8():
    out = []
    severities = [("base", 5), ("obfuscated", 4), ("worst", 3)]
    country_pool = ["IN", "AE", "SG", "GB", "US", "HK", "MU", "PA", "KY", "TR"]
    for sev, n_scenarios in severities:
        for i in range(n_scenarios):
            gid = f"T08_{sev[:1].upper()}{i+1:04d}"
            chain_len = {"base": 4, "obfuscated": 5, "worst": 7}[sev]
            chain = choose_accounts(chain_len + 1, predicate=None)
            if len(chain) < 4:
                continue

            dt0 = random_date_between()
            amount = random.uniform(100000, 850000 if sev != "worst" else 2500000)
            if sev == "worst":
                amount *= random.uniform(1.1, 1.8)

            countries = random.sample(country_pool, min(chain_len + 1, len(country_pool)))
            if countries[0] != "IN":
                countries[0] = "IN"
            if sev == "worst" and "AE" not in countries:
                countries[1] = "AE"

            for h in range(chain_len):
                sender = chain[h]
                receiver = chain[h + 1]
                hop_dt = dt0 + timedelta(days=h, minutes=random.randint(0, 240))
                hop_amount = amount * (0.98 ** h) * random.uniform(0.96, 1.0)

                out.append(build_row(
                    sender_acct=sender,
                    receiver_acct=receiver,
                    amount=hop_amount,
                    dt=random_time_on_date(hop_dt, preferred_hour=random.choice([9, 10, 11, 13, 16, 18])),
                    typology="Multi-Bank, Multi-Jurisdiction Chains",
                    group_id=gid,
                    channel=random.choice(["NEFT", "RTGS", "Internet Banking", "IMPS"]),
                    dr_cr="Dr",
                    sender_country=countries[h % len(countries)],
                    receiver_country=countries[(h + 1) % len(countries)],
                    is_aml=1,
                    device_mode="shared" if sev == "worst" else "sender",
                    ip_mode="shared" if sev == "worst" else "sender",
                    extra_updates={
                        "counterparty_branch_ifsc_swift": f"SWFT-{countries[(h + 1) % len(countries)]}-{random.randint(100,999)}",
                        "non_face_to_face_flag": "Y" if sev != "base" else "N",
                        "vpn_flag": "Y" if sev == "worst" else "N",
                    }
                ))

            # Worst case: close the loop back to the origin via a different jurisdiction
            if sev == "worst":
                sender = chain[-1]
                receiver = chain[0]
                hop_dt = dt0 + timedelta(days=chain_len, minutes=random.randint(10, 480))
                out.append(build_row(
                    sender_acct=sender,
                    receiver_acct=receiver,
                    amount=amount * random.uniform(0.90, 0.99),
                    dt=random_time_on_date(hop_dt, preferred_hour=random.choice([8, 12, 19, 22])),
                    typology="Multi-Bank, Multi-Jurisdiction Chains",
                    group_id=gid,
                    channel=random.choice(["SWIFT", "RTGS", "NEFT", "Internet Banking"]),
                    dr_cr="Dr",
                    sender_country=random.choice(["GB", "SG", "HK", "AE"]),
                    receiver_country="IN",
                    is_aml=1,
                    device_mode="shared",
                    ip_mode="shared",
                    extra_updates={
                        "counterparty_branch_ifsc_swift": f"SWFT-IN-{random.randint(100,999)}",
                        "vpn_flag": "Y",
                        "emulator_flag": "Y",
                    }
                ))
    return out



In [9]:
# ------------------------------------------------------------
# Typology 11: Third-Party Payment Webs
# ------------------------------------------------------------
def inject_typology_11():
    out = []
    severities = [("base", 5), ("obfuscated", 4), ("worst", 3)]
    for sev, n_scenarios in severities:
        for i in range(n_scenarios):
            gid = f"T11_{sev[:1].upper()}{i+1:04d}"
            business_pool = choose_accounts(20, predicate=is_business_profile)
            anchor_pool = business_pool if business_pool else choose_accounts(20)
            if len(anchor_pool) < 3:
                continue

            anchor = random.choice(anchor_pool)
            third_parties = choose_accounts(6 if sev != "worst" else 10, exclude=[anchor])
            if len(third_parties) < 4:
                continue

            intermediaries = choose_accounts(2 if sev != "worst" else 4, exclude=[anchor] + third_parties)
            if not intermediaries:
                intermediaries = third_parties[:1]

            dt0 = random_date_between()
            web_txns = {"base": 10, "obfuscated": 16, "worst": 24}[sev]

            for t in range(web_txns):
                if sev == "worst" and t % 4 == 0 and intermediaries:
                    sender = anchor
                    receiver = random.choice(intermediaries + third_parties)
                elif t % 5 in (0, 1):
                    sender = anchor
                    receiver = random.choice(third_parties)
                else:
                    sender = random.choice(third_parties)
                    receiver = random.choice(intermediaries + [anchor])

                amt = random.uniform(
                    15000 if sev == "base" else 5000,
                    75000 if sev != "worst" else 250000
                )
                hop_dt = dt0 + timedelta(days=random.randint(0, 12), hours=random.randint(0, 6), minutes=random.randint(0, 59))
                out.append(build_row(
                    sender_acct=sender,
                    receiver_acct=receiver,
                    amount=amt,
                    dt=random_time_on_date(hop_dt, preferred_hour=random.choice([9, 10, 11, 14, 16, 19])),
                    typology="Third-Party Payment Webs",
                    group_id=gid,
                    channel=random.choice(["NEFT", "IMPS", "UPI", "Internet Banking"]),
                    dr_cr="Dr",
                    is_aml=1,
                    device_mode="shared" if sev == "worst" else "sender",
                    ip_mode="shared" if sev == "worst" else "sender",
                    extra_updates={
                        "non_face_to_face_flag": "Y" if sev != "base" else "N",
                        "counterparty_name": f"Third Party {t+1:02d}",
                    }
                ))
    return out



In [10]:
# ------------------------------------------------------------
# Typology 16: Money Mule Networks
# ------------------------------------------------------------
def inject_typology_16():
    out = []
    severities = [("base", 5), ("obfuscated", 4), ("worst", 3)]
    for sev, n_scenarios in severities:
        for i in range(n_scenarios):
            gid = f"T16_{sev[:1].upper()}{i+1:04d}"
            controller_pool = choose_accounts(15, predicate=is_business_profile)
            if not controller_pool:
                controller_pool = choose_accounts(15)
            if len(controller_pool) < 3:
                continue

            controller = random.choice(controller_pool)
            recruiters = choose_accounts(2 if sev != "worst" else 4, exclude=[controller])
            mules = choose_accounts(6 if sev != "worst" else 10, exclude=[controller] + recruiters)
            exits = choose_accounts(2 if sev != "worst" else 4, exclude=[controller] + recruiters + mules)

            if len(mules) < 4:
                continue
            if not exits:
                exits = choose_accounts(2, exclude=[controller] + recruiters + mules)

            dt0 = random_date_between()
            shared_device = hashlib.md5(f"MULE|{gid}".encode()).hexdigest()[:32] if sev != "base" else None
            shared_ip = f"103.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}" if sev == "worst" else None

            # Controller funds recruiters / mules
            for idx, mule in enumerate(mules):
                feeder = controller if idx % 2 == 0 or not recruiters else random.choice([controller] + recruiters)
                incoming = random.uniform(25000, 150000 if sev != "worst" else 450000)
                dt_in = dt0 + timedelta(days=random.randint(0, 5), hours=random.randint(8, 20), minutes=random.randint(0, 59))
                out.append(build_row(
                    sender_acct=feeder,
                    receiver_acct=mule,
                    amount=incoming,
                    dt=random_time_on_date(dt_in, preferred_hour=random.choice([9, 10, 11, 13, 15, 18, 20])),
                    typology="Money Mule Networks",
                    group_id=gid,
                    channel=random.choice(["NEFT", "IMPS", "UPI", "RTGS"]),
                    dr_cr="Cr",
                    is_aml=1,
                    device_mode="shared" if shared_device else "sender",
                    shared_device_id=shared_device,
                    ip_mode="shared" if shared_ip else "sender",
                    shared_ip=shared_ip,
                    extra_updates={
                        "non_face_to_face_flag": "Y" if sev != "base" else "N",
                        "vpn_flag": "Y" if sev == "worst" else "N",
                        "emulator_flag": "Y" if sev == "worst" else "N",
                    }
                ))

                # Mule quickly forwards to exit nodes / cash out
                exits_pick = exits if exits else [controller]
                out_amt = incoming * random.uniform(0.90, 0.995)
                dt_out = dt_in + timedelta(minutes=random.randint(5, 180))

                channel = random.choice(["ATM", "Branch Cash", "UPI", "IMPS"]) if sev != "base" else random.choice(["UPI", "IMPS", "NEFT"])
                out.append(build_row(
                    sender_acct=mule,
                    receiver_acct=random.choice(exits_pick),
                    amount=out_amt,
                    dt=random_time_on_date(dt_out, preferred_hour=min(dt_out.hour, 23)),
                    typology="Money Mule Networks",
                    group_id=gid,
                    channel=channel,
                    dr_cr="Dr",
                    is_aml=1,
                    balance_before=random.uniform(50, 1500),
                    balance_after=random.uniform(0, 400) if channel in {"ATM", "Branch Cash"} else random.uniform(100, 1000),
                    device_mode="shared" if sev == "worst" else "sender",
                    shared_device_id=shared_device,
                    ip_mode="shared" if sev == "worst" else "sender",
                    shared_ip=shared_ip,
                    extra_updates={
                        "cash_flag": "Y" if channel in {"ATM", "Branch Cash"} else "N",
                        "non_face_to_face_flag": "Y" if sev != "base" else "N",
                    }
                ))

                if sev == "worst" and random.random() < 0.35:
                    # Extra hop inside the mule ring before cash-out
                    mid = random.choice(mules)
                    if mid != mule:
                        hop_dt = dt_out + timedelta(minutes=random.randint(3, 45))
                        out.append(build_row(
                            sender_acct=mule,
                            receiver_acct=mid,
                            amount=out_amt * random.uniform(0.40, 0.75),
                            dt=random_time_on_date(hop_dt, preferred_hour=min(hop_dt.hour, 23)),
                            typology="Money Mule Networks",
                            group_id=gid,
                            channel=random.choice(["UPI", "IMPS", "NEFT"]),
                            dr_cr="Dr",
                            is_aml=1,
                            device_mode="shared",
                            shared_device_id=shared_device,
                            ip_mode="shared",
                            shared_ip=shared_ip,
                            extra_updates={
                                "non_face_to_face_flag": "Y",
                                "vpn_flag": "Y",
                            }
                        ))
    return out



In [11]:
# ------------------------------------------------------------
# Generate the new typology rows
# ------------------------------------------------------------
new_rows = []
new_rows += inject_typology_6()
new_rows += inject_typology_7()
new_rows += inject_typology_8()
new_rows += inject_typology_11()
new_rows += inject_typology_16()

new_df = pd.DataFrame(new_rows)

# Keep only columns that exist in the source, and add any missing ones if needed
for col in df.columns:
    if col not in new_df.columns:
        new_df[col] = ""

new_df = new_df[df.columns]

# Optional: sort by date/time so the injected rows blend into the timeline
combined = pd.concat([df, new_df], ignore_index=True)

def _sort_key(row):
    d = _parse_dt(row.get("datestamp", ""))
    t = _norm(row.get("timestamp", "00:00:00"))
    if d is None:
        return pd.Timestamp("1900-01-01")
    try:
        dt = datetime.strptime(f"{row.get('datestamp', '')} {t}", "%d/%m/%Y %H:%M:%S")
    except ValueError:
        try:
            dt = datetime.strptime(f"{row.get('datestamp', '')} {t}", "%d-%m-%Y %H:%M:%S")
        except ValueError:
            dt = d
    return pd.Timestamp(dt)

combined["__sort_dt"] = combined.apply(_sort_key, axis=1)
combined = combined.sort_values(["__sort_dt", "transaction_id"], kind="mergesort").drop(columns=["__sort_dt"])

# Save output
combined.to_csv(OUTPUT_CSV, index=False)

# Simple report
print(f"Saved: {OUTPUT_CSV}")
print(f"Base rows: {len(df):,}")
print(f"Injected rows: {len(new_df):,}")
print(f"Final rows: {len(combined):,}")
print("\nInjected AML counts by typology:")
print(combined.loc[combined["is_aml"].astype(str) == "1", "aml_typology"].value_counts().head(20))

Saved: transactions_final_with_typologies_6_7_8_11_16.csv
Base rows: 650,100
Injected rows: 760
Final rows: 650,860

Injected AML counts by typology:
aml_typology
Funnel Account Network                           19370
Structuring (Smurfing)                           16400
Rapid Multi-Hop Layering                         13103
Circular Transaction Loop                         9604
Pass-Through Transit Hub                          6534
Third-Party Payment Webs                           186
Burst Activity After Dormancy in a Cluster         176
Money Mule Networks                                174
High-Velocity In/Out with Low Closing Balance      124
Multi-Bank, Multi-Jurisdiction Chains               64
Name: count, dtype: int64
